# SE3: Model train, validate, and interpret code

This notebook consolidates the full SE3 submission into a single file based on the uploaded data files:

- `/mnt/data/Estimated hourly.xlsx`
- `/mnt/data/Observed daily.xlsx`

It includes:

1. data loading
2. hourly-to-daily aggregation
3. merge with observed daily values
4. statistical model training
5. coefficient extraction and fit metrics
6. export of outputs
7. README-style workflow documentation
8. Data and Methods / Results / Discussion starter text

The workflow is consistent with the project README, which states that simulated hourly releases are aggregated to daily outflows for evaluation against observed daily plant outflows.


## Workflow summary

1. Read hourly simulated plant outputs and observed daily plant outputs.
2. Harmonize plant names across files.
3. Aggregate simulated hourly outputs to daily mean values.
4. Remove incomplete edge days.
5. Merge daily simulated outputs with daily observed outputs by date.
6. Reshape to long format for plant-level modeling.
7. Fit one OLS model per plant
8. Calculate training and validation metrics.
9. Export outputs for the SE3 submission.


In [6]:
pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


In [13]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

hourly_path = Path("Estimated hourly.xlsx")
observed_path = Path("Observed daily.xlsx")

outdir = Path("SE3_all_in_one_outputs")
(outdir / "model_data" / "evaluation").mkdir(parents=True, exist_ok=True)
(outdir / "model_data" / "statistical_model").mkdir(parents=True, exist_ok=True)

PLANTS = ["Jylhämä", "Nuojua", "Utanen", "pälli", "pyhäkoski", "montta", "Merikoksi"]

hourly_path, observed_path, outdir


(PosixPath('Estimated hourly.xlsx'),
 PosixPath('Observed daily.xlsx'),
 PosixPath('SE3_all_in_one_outputs'))

## 1) Read inputs and harmonize plant names

In [2]:
hourly = pd.read_excel(hourly_path)
observed = pd.read_excel(observed_path)

hourly["Date"] = pd.to_datetime(hourly["Date"])
observed["Date"] = pd.to_datetime(observed["Date"])

hourly = hourly.rename(columns={
    "Pälli": "pälli",
    "Pyhäkoski": "pyhäkoski",
    "Montta": "montta",
})

print("Hourly shape:", hourly.shape)
print("Observed shape:", observed.shape)
print("Hourly columns:", hourly.columns.tolist())
print("Observed columns:", observed.columns.tolist())

hourly.head()


Hourly shape: (87672, 8)
Observed shape: (3653, 8)
Hourly columns: ['Date', 'Jylhämä', 'Nuojua', 'Utanen', 'pälli', 'pyhäkoski', 'montta', 'Merikoksi']
Observed columns: ['Date', 'Jylhämä', 'Nuojua', 'Utanen', 'pälli', 'pyhäkoski', 'montta', 'Merikoksi']


,Date,Jylhämä,Nuojua,Utanen,pälli,pyhäkoski,montta,Merikoksi
0,2015-01-01 00:00:00.000,0.057,0.098,0.147,0.193,0.206,0.18,0.22
1,2015-01-01 01:00:00.000,0.057,0.098,0.147,0.193,0.206,0.18,0.22
2,2015-01-01 01:59:59.990,0.057,0.098,0.147,0.193,0.206,0.18,0.22
3,2015-01-01 02:59:59.985,0.057,0.098,0.147,0.193,0.206,0.18,0.22
4,2015-01-01 03:59:59.980,0.057,0.098,0.147,0.193,0.206,0.18,0.22


## 2) Aggregate hourly simulated outputs to daily values

Because the variable behaves like discharge or outflow, the hourly modeled values are aggregated to **daily mean** values before comparison with daily observations.



In [3]:
hourly["day"] = hourly["Date"].dt.floor("D")

counts = hourly.groupby("day").size().reset_index(name="n_hours")
counts.head(), counts.tail()


(         day  n_hours
 0 2015-01-01       25
 1 2015-01-02       24
 2 2015-01-03       24
 3 2015-01-04       24
 4 2015-01-05       24,
             day  n_hours
 3648 2024-12-27       24
 3649 2024-12-28       24
 3650 2024-12-29       24
 3651 2024-12-30       24
 3652 2024-12-31       23)

In [4]:
daily_sim = (
    hourly.groupby("day", as_index=False)[PLANTS]
    .mean()
    .rename(columns={"day": "Date"})
)

daily_sim = daily_sim.merge(counts, left_on="Date", right_on="day", how="left").drop(columns="day")
daily_sim_full = daily_sim[daily_sim["n_hours"] == 24].copy()

print("Daily simulated (all days):", daily_sim.shape)
print("Daily simulated (full 24h days only):", daily_sim_full.shape)

daily_sim_full.head()


Daily simulated (all days): (3653, 9)
Daily simulated (full 24h days only): (3651, 9)


,Date,Jylhämä,Nuojua,Utanen,pälli,pyhäkoski,montta,Merikoksi,n_hours
1,2015-01-02,0.118338,0.185947,0.238650,0.354547,0.348134,0.367322,0.483067,24
2,2015-01-03,0.368287,0.405790,0.432897,0.462245,0.477463,0.487405,0.535680,24
3,2015-01-04,0.449553,0.505918,0.534591,0.569943,0.598146,0.594373,0.746133,24
4,2015-01-05,1.318066,1.332947,1.292808,1.318725,1.346668,1.359715,1.370141,24
5,2015-01-06,1.222307,1.265504,1.286883,1.296658,1.311898,1.330280,1.430107,24


## 3) Merge daily simulated values with daily observed values

In [5]:
wide = observed.merge(daily_sim_full, on="Date", suffixes=("_obs", "_sim"), how="inner")

print("Merged wide daily shape:", wide.shape)
print("Date range:", wide["Date"].min().date(), "to", wide["Date"].max().date())

wide.head()


Merged wide daily shape: (3651, 16)
Date range: 2015-01-02 to 2024-12-30


,Date,Jylhämä_obs,Nuojua_obs,Utanen_obs,pälli_obs,pyhäkoski_obs,montta_obs,Merikoksi_obs,Jylhämä_sim,Nuojua_sim,Utanen_sim,pälli_sim,pyhäkoski_sim,montta_sim,Merikoksi_sim,n_hours
0,2015-01-02,0.2268,0.268812,0.299664,0.307188,0.321588,0.314712,0.297684,0.118338,0.185947,0.238650,0.354547,0.348134,0.367322,0.483067,24
1,2015-01-03,0.3924,0.430200,0.401868,0.421632,0.432972,0.419652,0.403452,0.368287,0.405790,0.432897,0.462245,0.477463,0.487405,0.535680,24
2,2015-01-04,0.5328,0.564192,0.546228,0.546696,0.551700,0.552276,0.505692,0.449553,0.505918,0.534591,0.569943,0.598146,0.594373,0.746133,24
3,2015-01-05,1.2456,1.315116,1.212480,1.207224,1.201248,1.191204,0.968400,1.318066,1.332947,1.292808,1.318725,1.346668,1.359715,1.370141,24
4,2015-01-06,1.0872,1.140300,1.140408,1.201176,1.193724,1.207044,1.270692,1.222307,1.265504,1.286883,1.296658,1.311898,1.330280,1.430107,24


## 4) Reshape to long format for plant-level modeling

In [6]:
obs_long = wide.melt(
    id_vars=["Date"],
    value_vars=[f"{p}_obs" for p in PLANTS],
    var_name="plant_obs",
    value_name="observed_daily_outflow",
)
obs_long["plant"] = obs_long["plant_obs"].str.replace("_obs", "", regex=False)

sim_long = wide.melt(
    id_vars=["Date"],
    value_vars=[f"{p}_sim" for p in PLANTS],
    var_name="plant_sim",
    value_name="simulated_daily_outflow",
)
sim_long["plant"] = sim_long["plant_sim"].str.replace("_sim", "", regex=False)

long_df = (
    obs_long[["Date", "plant", "observed_daily_outflow"]]
    .merge(sim_long[["Date", "plant", "simulated_daily_outflow"]], on=["Date", "plant"], how="inner")
    .sort_values(["plant", "Date"])
    .reset_index(drop=True)
)

long_df["doy"] = long_df["Date"].dt.dayofyear
long_df["sin_doy"] = np.sin(2 * np.pi * long_df["doy"] / 365.25)
long_df["cos_doy"] = np.cos(2 * np.pi * long_df["doy"] / 365.25)

print("Long panel shape:", long_df.shape)
long_df.head()


Long panel shape: (25557, 7)


,Date,plant,observed_daily_outflow,simulated_daily_outflow,doy,sin_doy,cos_doy
0,2015-01-02,Jylhämä,0.2268,0.118338,2,0.034398,0.999408
1,2015-01-03,Jylhämä,0.3924,0.368287,3,0.051584,0.998669
2,2015-01-04,Jylhämä,0.5328,0.449553,4,0.068755,0.997634
3,2015-01-05,Jylhämä,1.2456,1.318066,5,0.085906,0.996303
4,2015-01-06,Jylhämä,1.0872,1.222307,6,0.103031,0.994678


## 5) Define fit metrics

In [7]:
def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.sqrt(np.mean((y - yhat) ** 2)))

def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(np.abs(y - yhat)))

def bias(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return float(np.mean(yhat - y))

def r2_manual(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    ss_res = np.sum((y - yhat) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot != 0 else np.nan

def nse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    denom = np.sum((y - np.mean(y)) ** 2)
    num = np.sum((y - yhat) ** 2)
    return float(1 - num / denom) if denom != 0 else np.nan

def kge(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    r = np.corrcoef(y, yhat)[0, 1]
    alpha = np.std(yhat, ddof=1) / np.std(y, ddof=1)
    beta = np.mean(yhat) / np.mean(y)
    return float(1 - np.sqrt((r - 1) ** 2 + (alpha - 1) ** 2 + (beta - 1) ** 2))

def adjusted_r2(r2, n, p):
    if n <= p + 1:
        return np.nan
    return float(1 - (1 - r2) * (n - 1) / (n - p - 1))

def standardize_series(s):
    s = pd.Series(s)
    std = s.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.mean()) / std

def calc_metrics(df, pred_col="predicted", obs_col="observed_daily_outflow", p=3):
    y = df[obs_col].values
    yhat = df[pred_col].values
    r2 = r2_manual(y, yhat)
    return {
        "n": int(len(df)),
        "R2": r2,
        "Adj_R2": adjusted_r2(r2, len(df), p),
        "RMSE": rmse(y, yhat),
        "MAE": mae(y, yhat),
        "Bias": bias(y, yhat),
        "NSE": nse(y, yhat),
        "KGE": kge(y, yhat),
    }


## 6) Fit one interpretable OLS model per plant

In [8]:
coef_rows = []
metric_rows = []
pred_rows = []
summary_texts = []

for plant in PLANTS:
    df = long_df[long_df["plant"] == plant].sort_values("Date").copy()
    split_ix = int(len(df) * 0.8)
    train = df.iloc[:split_ix].copy()
    valid = df.iloc[split_ix:].copy()

    formula = "observed_daily_outflow ~ simulated_daily_outflow + sin_doy + cos_doy"
    model = smf.ols(formula, data=train).fit(cov_type="HC3")

    train["predicted"] = model.predict(train)
    valid["predicted"] = model.predict(valid)

    train["split"] = "train"
    valid["split"] = "validation"

    pred_rows.append(train)
    pred_rows.append(valid)

    train_metrics = calc_metrics(train)
    valid_metrics = calc_metrics(valid)

    metric_rows.append({"plant": plant, "split": "train", **train_metrics})
    metric_rows.append({"plant": plant, "split": "validation", **valid_metrics})

    params = model.params
    conf = model.conf_int()
    bse = model.bse
    pvals = model.pvalues
    tvals = model.tvalues

    train_std = train.copy()
    train_std["y_std"] = standardize_series(train_std["observed_daily_outflow"])
    train_std["x1_std"] = standardize_series(train_std["simulated_daily_outflow"])
    train_std["x2_std"] = standardize_series(train_std["sin_doy"])
    train_std["x3_std"] = standardize_series(train_std["cos_doy"])

    std_model = smf.ols("y_std ~ x1_std + x2_std + x3_std", data=train_std).fit(cov_type="HC3")
    std_lookup = {
        "Intercept": np.nan,
        "simulated_daily_outflow": std_model.params.get("x1_std", np.nan),
        "sin_doy": std_model.params.get("x2_std", np.nan),
        "cos_doy": std_model.params.get("x3_std", np.nan),
    }

    for term in params.index:
        coef_rows.append({
            "plant": plant,
            "term": term,
            "coefficient": float(params[term]),
            "std_error": float(bse[term]),
            "t_value": float(tvals[term]),
            "p_value": float(pvals[term]),
            "ci_lower": float(conf.loc[term, 0]),
            "ci_upper": float(conf.loc[term, 1]),
            "standardized_beta": float(std_lookup.get(term, np.nan)) if pd.notna(std_lookup.get(term, np.nan)) else np.nan,
        })

    summary_texts.append(f"\n{'='*80}\nPLANT: {plant}\n{'='*80}\n")
    summary_texts.append(model.summary().as_text())

coefficients = pd.DataFrame(coef_rows)
fit_metrics = pd.DataFrame(metric_rows)
predictions = pd.concat(pred_rows, ignore_index=True)

coefficients.head(), fit_metrics.head()


(     plant                     term  coefficient  std_error    t_value  \
 0  Jylhämä                Intercept     0.262225   0.011102  23.619307   
 1  Jylhämä  simulated_daily_outflow     0.716670   0.010497  68.276008   
 2  Jylhämä                  sin_doy     0.037158   0.006708   5.539490   
 3  Jylhämä                  cos_doy     0.104656   0.007151  14.634456   
 4   Nuojua                Intercept     0.248471   0.011625  21.373960   
 
          p_value  ci_lower  ci_upper  standardized_beta  
 0  2.441217e-123  0.240465  0.283984                NaN  
 1   0.000000e+00  0.696097  0.737243           0.775451  
 2   3.033541e-08  0.024011  0.050305           0.057714  
 3   1.693237e-48  0.090640  0.118673           0.162442  
 4  2.334690e-101  0.225686  0.271255                NaN  ,
      plant       split     n        R2    Adj_R2      RMSE       MAE  \
 0  Jylhämä       train  2920  0.711673  0.711376  0.244538  0.187296   
 1  Jylhämä  validation   731  0.226065  0.2228

## 7) Save SE3 outputs

In [14]:
daily_sim_full.to_csv(outdir / "model_data" / "evaluation" / "daily_simulated_from_hourly.csv", index=False)
wide.to_csv(outdir / "model_data" / "evaluation" / "daily_comparison_wide.csv", index=False)
long_df.to_csv(outdir / "model_data" / "evaluation" / "daily_comparison_long.csv", index=False)

coefficients.to_csv(outdir / "model_data" / "statistical_model" / "coefficients.csv", index=False)
fit_metrics.to_csv(outdir / "model_data" / "statistical_model" / "fit_metrics.csv", index=False)
predictions.to_csv(outdir / "model_data" / "statistical_model" / "predictions.csv", index=False)

with open(outdir / "model_data" / "statistical_model" / "model_summary.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(summary_texts))

manifest = {
    "inputs": {
        "hourly": str(hourly_path),
        "observed": str(observed_path),
    },
    "outputs": {
        "daily_simulated_from_hourly": str(outdir / "model_data" / "evaluation" / "daily_simulated_from_hourly.csv"),
        "daily_comparison_wide": str(outdir / "model_data" / "evaluation" / "daily_comparison_wide.csv"),
        "daily_comparison_long": str(outdir / "model_data" / "evaluation" / "daily_comparison_long.csv"),
        "coefficients": str(outdir / "model_data" / "statistical_model" / "coefficients.csv"),
        "fit_metrics": str(outdir / "model_data" / "statistical_model" / "fit_metrics.csv"),
        "predictions": str(outdir / "model_data" / "statistical_model" / "predictions.csv"),
        "model_summary": str(outdir / "model_data" / "statistical_model" / "model_summary.txt"),
    },
    "model_formula": "observed_daily_outflow ~ simulated_daily_outflow + sin_doy + cos_doy",
    "estimator": "OLS",
    "robust_se": "HC3",
    "split": "chronological 80/20",
}

with open(outdir / "model_data" / "statistical_model" / "model_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved all outputs to:", outdir)


Saved all outputs to: SE3_all_in_one_outputs


## 8) Main SE3 evidence tables

In [10]:
validation_metrics = fit_metrics[fit_metrics["split"] == "validation"].sort_values("plant").reset_index(drop=True)
validation_metrics


,plant,split,n,R2,Adj_R2,RMSE,MAE,Bias,NSE,KGE
0,Jylhämä,validation,731,0.226065,0.222871,0.392821,0.295335,-0.019577,0.226065,0.596738
1,Merikoksi,validation,731,0.226116,0.222922,0.405220,0.310478,0.013358,0.226116,0.603667
2,Nuojua,validation,731,0.223730,0.220527,0.403902,0.303939,-0.012763,0.223730,0.594437
3,Utanen,validation,731,0.260564,0.257513,0.413031,0.316731,-0.051853,0.260564,0.588258
4,montta,validation,731,0.214971,0.211731,0.395275,0.296231,-0.016341,0.214971,0.604300
5,pyhäkoski,validation,731,0.236333,0.233181,0.395187,0.297538,-0.018827,0.236333,0.604897
6,pälli,validation,731,0.248942,0.245842,0.395271,0.297965,-0.005328,0.248942,0.603579


In [11]:
coefficients[coefficients["term"] == "simulated_daily_outflow"].sort_values("plant").reset_index(drop=True)


,plant,term,coefficient,std_error,t_value,p_value,ci_lower,ci_upper,standardized_beta
0,Jylhämä,simulated_daily_outflow,0.716670,0.010497,68.276008,0.0,0.696097,0.737243,0.775451
1,Merikoksi,simulated_daily_outflow,0.786021,0.011417,68.845707,0.0,0.763643,0.808398,0.794631
2,Nuojua,simulated_daily_outflow,0.745872,0.010619,70.241600,0.0,0.725060,0.766684,0.784687
3,Utanen,simulated_daily_outflow,0.752518,0.010471,71.864613,0.0,0.731994,0.773041,0.795078
4,montta,simulated_daily_outflow,0.758100,0.010292,73.658325,0.0,0.737928,0.778273,0.807171
5,pyhäkoski,simulated_daily_outflow,0.751969,0.010216,73.607874,0.0,0.731947,0.771992,0.806948
6,pälli,simulated_daily_outflow,0.748331,0.010122,73.930823,0.0,0.728492,0.768170,0.801764
